# Appendix_1: Mathematics of the scalar grating solver

The scalar grating solver turns a material profile into reflected and transmitted complex amplitudes. This appendix follows that calculation in `wave_optics/scalar_grating.py`, linking the physical model to its numerical representation.

**Medium and wavelength convention.** `ScalarGratingConfiguration.wavelength` is **vacuum** wavelength $\lambda_0$. The solver separately specifies incident, output, ridge and groove indices; its defaults are $n_i=n_t=n_{\rm groove}=1$ and $n_{\rm ridge}=1.5$. The absorber extension uses a complex layer index with air exteriors. Exterior wavenumbers are $n_i k_0$ and $n_t k_0$, not universally $k_0$. See the [course medium convention](0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).

This appendix follows the [actual implementation](wave_optics/scalar_grating.py), from the scalar Helmholtz equation to the arrays returned by the solver. Read it after [0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) and [0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb). Familiarity with complex exponentials, Fourier series, matrix multiplication and eigenvectors is helpful; the physical meaning accompanies each calculation.

The worked example uses the real-index configuration defined in `scalar_grating.py`. Section 12 connects the same matching engine to the absorbing grating used in [0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb)--[0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb).

The calculation proceeds from the material profile through its Fourier coefficients and coupled Helmholtz equations, then to layer eigenmodes and interface matching. The resulting complex diffraction coefficients determine the field and power.

Run the code cells in order in a fresh kernel. They use the existing backend and inspect its result through short algebraic checks. No notebook from the main sequence needs to have been run first.


**Runnable Python counterparts:** [A1_1_scalar_fmm_grating.py numerical explorer](Wave_solver_py_scripts/numerical_explorer/A1_1_scalar_fmm_grating.py) reuses this solver with physical-parameter controls, basis cutoff M, full versus propagating-output plots and optional convergence checks. The [standard Python script](Wave_solver_py_scripts/scripts/A1_1_scalar_fmm_grating.py) provides a direct calculation and static plots. Edit the parameter block and run the whole file. [Download either collection](Downloads.md).


## 1. What physical problem is being solved?

Take the physical field to be $u(x,z,t)=\operatorname{Re}\{U(x,z)e^{-i\omega t}\}$. Its complex amplitude satisfies

$$
\boxed{\frac{\partial^2 U}{\partial x^2}+\frac{\partial^2 U}{\partial z^2}
+k_0^2\varepsilon(x,z)U=0,}
\qquad k_0=\frac{2\pi}{\lambda_0},\qquad \varepsilon=n^2.
$$

Here $\varepsilon$ is relative permittivity in the scalar model. The grating is periodic in $x$, invariant in $y$, and occupies $-h<z<0$. A unit plane wave enters from above, travelling toward **negative $z$**:

$$U_i(z)=e^{-ik_0n_i z},\qquad U_i(0)=1.$$

| Region | Coordinates | Prescription |
| --- | --- | --- |
| Incident medium | $z\geq0$ | Index $n_i$; known incidence plus unknown reflection |
| Periodic layer | $-h<z<0$ | Period $d$, thickness $h$, ridge fraction $D_{\mathrm{ridge}}$, indices $n_r,n_g$ |
| Output medium | $z\leq-h$ | Index $n_o$; only outgoing or decaying transmitted waves |

The scalar interface conditions are continuity of $U$ and $\partial_zU$. This defines a scalar boundary-value problem, not a general vector Maxwell treatment of both polarizations. There is one layer and normal incidence.

**In the code:** `ScalarGratingConfiguration` validates positive real indices and lengths, $0\leq D_{\mathrm{ridge}}\leq1$, and integer $M\geq0$. The ridge is centred on $x=0$ in each period. All lengths share one unit; here we use micrometres and a **vacuum** wavelength.

**Predict:** if every index is the same, should there be any reflected or higher-order field? We check this in Section 11.

**Field convention:** the exit is $z_e=-h$. Write $U_{\mathrm{exit}}(x)=U(x,z_e)$ and use $C_m(z_e)$ for transmitted field coefficients. Their vector is $\mathbf C_{\mathrm{exit}}$. The backend retains `solution.transmission` and its internal array names; these store that same vector. The symbol $t_m$ is reserved in the course for Fourier coefficients of a prescribed thin-mask transmission.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

_support_dir = next((base for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (base / 'wave_optics' / 'scalar_grating.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this appendix within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))

import numpy as np
from numpy.testing import assert_allclose
from scipy import linalg
import matplotlib.pyplot as plt
from IPython.display import display
from wave_optics.scalar_grating import (
    ScalarGratingConfiguration, solve_scalar_grating, _layer_basis,
)

config = ScalarGratingConfiguration(M=10)
solution = solve_scalar_grating(config)
k0 = 2 * np.pi / config.wavelength
orders = solution.orders
N = orders.size
G = 2 * np.pi / config.period
kx_over_k0 = solution.kx / k0
L = k0 * config.thickness

# Internal arrays are inspected only to explain this implementation.
# Applications normally use harmonics/reconstruct/at_reference_plane.
q, V = solution._q, solution._vectors
a, b = solution._amplitudes

def finish_figure(fig):
    display(fig)
    plt.close(fig)  # Repeated execution does not retain Matplotlib figures.

print(f'{N} field harmonics: m = {orders[0]} ... {orders[-1]}')
print(f'lambda0={config.wavelength:g} um, d={config.period:g} um, h={config.thickness:g} um')
print(f'Layer: n_r={config.n_ridge:g}, n_g={config.n_groove:g}; D_ridge={config.duty_cycle:g}')


## 2. How does the rectangular material become Fourier coefficients?

Inside the layer, let $R(x)$ be one in the ridge, $-D_{\mathrm{ridge}}d/2\leq x<D_{\mathrm{ridge}}d/2$, and zero in the groove, repeated every period. Then

$$\varepsilon(x)=n_g^2+(n_r^2-n_g^2)R(x).$$

Use the Fourier convention

$$
\varepsilon(x)=\sum_{\ell=-\infty}^{\infty}\varepsilon_\ell e^{i\ell Gx},
\qquad
\varepsilon_\ell=\frac1d\int_{-d/2}^{d/2}\varepsilon(x)e^{-i\ell Gx}\,dx,
\qquad G=\frac{2\pi}{d}.
$$

The uniform background contributes only to $\ell=0$. The ridge contributes

$$\frac1d\int_{-D_{\mathrm{ridge}}d/2}^{D_{\mathrm{ridge}}d/2}e^{-i\ell Gx}\,dx=D_{\mathrm{ridge}}\,\operatorname{sinc}(\ell D_{\mathrm{ridge}}).$$

Therefore,

$$
\boxed{\varepsilon_\ell=n_g^2\delta_{\ell0}
+(n_r^2-n_g^2)D_{\mathrm{ridge}}\,\operatorname{sinc}(\ell D_{\mathrm{ridge}}),}
\qquad
\operatorname{sinc}(v)=\frac{\sin(\pi v)}{\pi v},\quad \operatorname{sinc}(0)=1.
$$

**In the code:** `permittivity_coefficients(p)` evaluates this expression; its argument `p` is the Fourier index $\ell$, not the period. `np.sinc` includes the factor $\pi$. The endpoints $D_{\mathrm{ridge}}=0,1$ are handled as uniform materials.

These are **material coefficients**, not field coefficients $F_m$ or $C_m$. `refractive_index` returns the exact prescribed piecewise profile. A finite Fourier reconstruction oscillates near its sharp edges.

**Explore:** change `material_terms` below. More terms sharpen the reconstruction but do not change the prescribed material or the solved example.


In [ ]:
material_terms = 20  # Try 4 or 40; this changes only the material illustration.
ell = np.arange(-material_terms, material_terms + 1)
x_material = np.linspace(-config.period/2, config.period/2, 801, endpoint=False)
eps_exact = config.refractive_index(x_material, -config.thickness/2)[0]**2
eps_coefficients = config.permittivity_coefficients(ell)
eps_series = eps_coefficients @ np.exp(1j * ell[:, None] * G * x_material)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].plot(x_material, eps_exact, color='black', lw=2, label='Prescribed material')
axes[0].plot(x_material, eps_series.real, color='#2866a3', label='Finite Fourier reconstruction')
axes[0].set(xlabel=r'$x$ ($\mu$m)', ylabel=r'$\varepsilon(x)$', title='Material, not optical field')
axes[0].legend(fontsize=9)
shown = abs(ell) <= 7
axes[1].stem(ell[shown], eps_coefficients[shown].real)
axes[1].set(xlabel=r'Material index $\ell$', ylabel=r'$\varepsilon_\ell$',
            title='Real coefficients of the centred, lossless profile')
for ax in axes:
    ax.grid(alpha=.15)
finish_figure(fig)
assert_allclose(config.permittivity_coefficients(0),
                (1-config.duty_cycle)*config.n_groove**2
                + config.duty_cycle*config.n_ridge**2)


## 3. Why do the field harmonics couple?

At normal incidence the field has the material's period. Retain $N=2M+1$ harmonics:

$$U_M(x,z)=\sum_{m=-M}^{M}F_m(z)e^{imGx},\qquad k_{x,m}=mG.$$

Differentiating twice in $x$ multiplies harmonic $m$ by $-k_{x,m}^2$. Multiplication by the material combines indices:

$$\varepsilon(x)U_M(x,z)
=\sum_\ell\sum_{p=-M}^{M}\varepsilon_\ell F_p(z)e^{i(\ell+p)Gx}.$$

To contribute to harmonic $m$, the indices must satisfy $\ell+p=m$. Projecting Helmholtz onto each retained harmonic gives

$$F_m''(z)-k_{x,m}^2F_m(z)
+k_0^2\sum_{p=-M}^{M}\varepsilon_{m-p}F_p(z)=0.$$

Collect the amplitudes into $\mathbf A=(F_{-M},\ldots,F_M)^{\mathsf T}$:

$$
\boxed{\mathbf A_{zz}+\mathbf Q\mathbf A=0,\qquad
Q_{mp}=k_0^2\varepsilon_{m-p}-k_{x,m}^2\delta_{mp}.}
$$

**Distinguish three descriptions:**

- **Fourier harmonics:** $F_m(z)$ are coupled inside the patterned material.
- **Layer eigenmodes:** diagonalizing the coupling matrix combines harmonics into modes that evolve independently within a uniform-in-depth layer. Interface matching sets their weights.
- **Exterior orders:** in a homogeneous region, the Fourier harmonics decouple and propagate or decay independently.

An off-diagonal entry says how one harmonic contributes to another's equation. A homogeneous material has only $\varepsilon_0$, so the equations decouple.

This is a **Fourier--Galerkin truncation**, used here as a Scalar Fourier Modal Method (FMM). The residual vanishes when projected onto the retained harmonics. It need not vanish point by point for finite $M$.

**Checkpoint:** why are material coefficients needed through $|\ell|=2M$ when the field retains only $|m|\leq M$?

:::{dropdown} Check your answer

The coupling uses $m-p$, whose largest magnitude is $2M$, for example at $m=M,p=-M$. Those extra material coefficients are not extra field orders.

:::

The complex modal functions are $F_m(z)$; the vector $\mathbf A$ below collects these functions to match the backend’s existing matrix notation. They are distinct from the nonnegative magnitudes $A_m=|C_m|$ used in the plane-wave lessons.


## 4. Why does the code use a dimensionless matrix?

Introduce downward depth and dimensionless thickness,

$$s=-k_0z,\qquad L=k_0h,\qquad
\bar{k}_{x,m}=\frac{k_{x,m}}{k_0}=\frac{m\lambda_0}{d}.$$

Since $\partial_z=-k_0\partial_s$, the layer equation becomes

$$
\boxed{\mathbf A_{ss}+\mathbf H\mathbf A=0,\qquad
H_{mp}=\varepsilon_{m-p}-\bar{k}_{x,m}^2\delta_{mp},\qquad
\mathbf Q=k_0^2\mathbf H.}
$$

| Mathematical object | Code name | Units |
| --- | --- | --- |
| $\mathbf H$ | Local `coupling` in `_solve_layer` | Dimensionless |
| $\mathbf Q$ | Returned `solution.coupling_matrix` | Inverse length squared |
| $s,L,\bar{k}_{x,m}$ | `depth`, `length`, `alpha` in the backend; `kx_over_k0` in the snippet | Dimensionless |
| $k_{x,m}$ | `solution.kx` | Inverse length |

The subtraction of $\bar{k}_{x,m}^2$ changes only the diagonal. Scaling removes an arbitrary length-unit factor from the eigensystem.

**In the code:** `orders[:, None] - orders[None, :]` constructs all differences $m-p$. The next cell reproduces that matrix, not a new scattering solve.

Here $\bar{k}_{x,m}=k_{x,m}/k_0=n\alpha_m$ in a homogeneous medium of index $n$; it is not the direction cosine $\alpha_m=k_{x,m}/k$. The depth $s=-k_0z$ is dimensionless, unlike Chapter 4’s physical depth $s=-z$. The snippet uses `kx_over_k0`; the backend retains the local variable name `alpha` for this vacuum-normalized quantity.


In [ ]:
E = config.permittivity_coefficients(orders[:, None] - orders[None, :])
H = E - np.diag(kx_over_k0**2)
assert_allclose(k0**2 * H, solution.coupling_matrix, rtol=1e-13, atol=1e-12)
assert_allclose(H, H.conj().T, atol=1e-14)
print(f'H shape: {H.shape}; boundary system will have {2*N} unknown modal weights.')
print(f'Largest material index used: {np.max(abs(orders[:, None]-orders[None, :]))}')

# Display a central block; H and the solver retain every order.
show = abs(orders) <= 3
shown_orders = orders[show]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for ax, matrix, title in zip(axes, (E, H), (r'Material coupling $E_{mp}=\varepsilon_{m-p}$',
                                          r'Layer operator $H=E-\mathrm{diag}(\bar{k}_{x,m}^2)$')):
    values = matrix[np.ix_(show, show)].real
    bound = max(np.max(abs(values)), 1e-12)
    picture = ax.imshow(values, cmap='RdBu_r', vmin=-bound, vmax=bound, origin='lower')
    ax.set(xticks=np.arange(len(shown_orders)), xticklabels=shown_orders,
           yticks=np.arange(len(shown_orders)), yticklabels=shown_orders,
           xlabel='Input harmonic n', ylabel='Equation m', title=title)
    fig.colorbar(picture, ax=ax, shrink=.85, label='Dimensionless entry')
finish_figure(fig)


## 5. What are the layer eigenmodes?

A single harmonic $e^{imGx}$ is generally not an independent mode inside the periodic layer. Seek combinations $\mathbf v_j$ satisfying

$$\mathbf H\mathbf v_j=q_j^2\mathbf v_j.$$

Put the eigenvectors in the columns of $\mathbf V$. With $\mathbf A(s)=\mathbf V\mathbf w(s)$, the coupled system becomes

$$
\mathbf V^{-1}\mathbf H\mathbf V=\operatorname{diag}(q_j^2),
\qquad w_j''(s)+q_j^2w_j(s)=0.
$$

Each column of $\mathbf V$ mixes Fourier harmonics into one layer mode. The mode index $j$ is **not** the diffraction-order index $m$. The dimensionless $q_j$ describes depth dependence; $k_0q_j$ has inverse-length units.

For real lossless material, $\varepsilon_{-\ell}=\varepsilon_\ell^*$, so $\mathbf H$ is Hermitian. `_solve_layer` uses `scipy.linalg.eigh`; then $\mathbf V^{-1}=\mathbf V^\dagger$. For passive absorption it uses `eig`, and that orthogonality cannot be assumed (Section 12).

The code takes $q_j=\sqrt{q_j^2}$ as a complex square root: a negative real eigenvalue gives a positive imaginary root. Evanescent modes are retained.


In [ ]:
# Compare operators, not individual eigenvector signs or ordering.
eigen_error = linalg.norm(H @ V - V * q**2) / (linalg.norm(H) * linalg.norm(V))
assert eigen_error < 1e-12
assert_allclose(V.conj().T @ V, np.eye(N), atol=1e-12)
print(f'Relative eigensystem residual: {eigen_error:.3e}')
print(f'Layer roots: {np.count_nonzero(q.imag == 0)} real, '
      f'{np.count_nonzero(q.imag > 0)} with positive imaginary part.')


## 6. Why use two boundary-anchored solutions?

A second-order equation needs two independent solutions. Away from cutoff, `_layer_basis` uses

$$f_j(s)=e^{iq_js},\qquad g_j(s)=e^{iq_j(L-s)},$$

$$f_j'(s)=iq_jf_j(s),\qquad g_j'(s)=-iq_jg_j(s).$$

The first is anchored at the top, the second at the bottom. If $q_j=i\kappa_j$, with $\kappa_j>0$, their magnitudes are $e^{-\kappa_js}$ and $e^{-\kappa_j(L-s)}$: neither exceeds one inside the layer. This avoids a transfer factor growing exponentially across a thick layer. The unknown modal weights themselves need not be at most one.

At **internal cutoff**, $q_j=0$, both exponentials become 1 and lose independence. For $|q_jL|<0.1$, the code instead uses the exact solutions

$$
f_j(s)=\cos(q_js),\qquad
g_j(s)=\frac{\sin(q_js)}{q_jL}
=\frac{s}{L}\operatorname{sinc}\!\left(\frac{q_js}{\pi}\right),
$$

$$f_j'(s)=-q_j\sin(q_js),\qquad g_j'(s)=\frac{\cos(q_js)}{L}.$$

Their cutoff limits are $f_j=1$, $g_j=s/L$, with derivatives $0$ and $1/L$. Both solutions survive. The threshold 0.1 is a numerical basis choice, not a physical boundary.

**Explore:** the left plot shows an evanescent pair; the right shows the exact cutoff pair. These examples use an illustrative $L=1$, independently of the solved grating.


In [ ]:
basis_depth = np.linspace(0, 1, 201)
f_ev, g_ev, _, _ = _layer_basis(np.array([4j]), basis_depth, 1.)
f_zero, g_zero, df_zero, dg_zero = _layer_basis(np.array([0j]), basis_depth, 1.)
assert_allclose(f_zero[:, 0], 1.)
assert_allclose(g_zero[:, 0], basis_depth)
assert_allclose(df_zero[:, 0], 0.)
assert_allclose(dg_zero[:, 0], 1.)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), constrained_layout=True)
axes[0].plot(basis_depth, abs(f_ev[:, 0]), label=r'$|f|$: anchored at top')
axes[0].plot(basis_depth, abs(g_ev[:, 0]), label=r'$|g|$: anchored at bottom')
axes[0].set(title=r'Evanescent mode: $q=4i$', ylabel='Basis magnitude')
axes[1].plot(basis_depth, f_zero[:, 0].real, label=r'$f=1$')
axes[1].plot(basis_depth, g_zero[:, 0].real, label=r'$g=s/L$')
axes[1].set(title=r'Cutoff mode: $q=0$', ylabel='Basis value')
for ax in axes:
    ax.set(xlabel=r'Downward depth $s/L$', ylim=(-.04, 1.12))
    ax.legend(fontsize=9)
    ax.grid(alpha=.15)
finish_figure(fig)


## 7. Which exterior waves are allowed?

In a homogeneous region $j=i,o$, define

$$
b_{j,m}=\sqrt{n_j^2-\bar{k}_{x,m}^2},\qquad
\beta_{j,m}=k_0b_{j,m},\qquad
\mathbf B_j=\operatorname{diag}(b_{j,m}).
$$

`_exterior_branch` chooses the positive real root for propagating orders and positive imaginary root for evanescent orders. An exactly grazing root is zero.

| Type | Root | Outgoing normal power |
| --- | --- | --- |
| Propagating | $\beta>0$ real | May be nonzero |
| Grazing | $\beta=0$ | Zero |
| Evanescent | $\beta=i\gamma$, $\gamma>0$ | Zero for the single outgoing/decaying exterior component |

The exterior fields are

$$
U(x,z)=e^{-i\beta_{i,0}z}
+\sum_m r_m e^{imGx+i\beta_{i,m}z},\qquad z\geq0,
$$

$$
U(x,z)=\sum_m C_m(z_e) e^{imGx-i\beta_{o,m}(z+h)},\qquad z\leq-h.
$$

Reflection is referenced to $z=0$ and transmission to $z=-h$. The reflected signed $k_z$ is $+\beta_i$; the transmitted signed $k_z$ is $-\beta_o$. Substituting $\beta=i\gamma$ confirms decay toward the corresponding infinity.

At grazing, the scalar equation also permits a linear solution in $z$; the bounded exterior choice keeps the constant solution. Some exact-threshold configurations nevertheless make the full outgoing problem nonunique; the solver reports this rather than assigning arbitrary coefficients.

**In the code:** values within `64 * eps * max(n**2, kx_over_k0**2)` of zero in the squared root are classified as grazing. Here `eps` is machine precision, not material permittivity.


## 8. How do the interfaces determine the unknown amplitudes?

Let $\mathbf e_0$ contain one at $m=0$ and zeros elsewhere. Its Python index is `M`, since the array starts at $m=-M$.

At the top, field continuity gives $\mathbf A(0)=\mathbf e_0+\mathbf r$. Derivative continuity in $s=-k_0z$ gives

$$\mathbf A_s(0)=i\mathbf B_i(\mathbf e_0-\mathbf r).$$

Eliminate $\mathbf r=\mathbf A(0)-\mathbf e_0$:

$$\boxed{\mathbf A_s(0)+i\mathbf B_i\mathbf A(0)=2i\mathbf B_i\mathbf e_0.}$$

At the bottom, $\mathbf A(L)=\mathbf C_{\mathrm{exit}}$ and $\mathbf A_s(L)=i\mathbf B_o\mathbf C_{\mathrm{exit}}$, so

$$\boxed{\mathbf A_s(L)-i\mathbf B_o\mathbf A(L)=0.}$$

These are mixed field-and-derivative, or **Robin**, conditions. The factor 2 comes from combining the known incident field and its derivative. There is no incidence from below.

Define $\mathbf F(s)=\operatorname{diag}(f_j(s))$ and $\mathbf G(s)=\operatorname{diag}(g_j(s))$. The layer field is

$$\mathbf A(s)=\mathbf V[\mathbf F(s)\mathbf a+\mathbf G(s)\mathbf b],$$

with two unknown modal-weight vectors, $\mathbf a,\mathbf b$. Substitution into the interface conditions gives a $2N\times2N$ system:

$$
\underbrace{\begin{pmatrix}
\mathbf V\mathbf F'_0+i\mathbf B_i\mathbf V\mathbf F_0&
\mathbf V\mathbf G'_0+i\mathbf B_i\mathbf V\mathbf G_0\\
\mathbf V\mathbf F'_L-i\mathbf B_o\mathbf V\mathbf F_L&
\mathbf V\mathbf G'_L-i\mathbf B_o\mathbf V\mathbf G_L
\end{pmatrix}}_{\mathbf K}
\underbrace{\begin{pmatrix}\mathbf a\\\mathbf b\end{pmatrix}}_{\mathbf c}
=
\underbrace{\begin{pmatrix}2i\mathbf B_i\mathbf e_0\\\mathbf0\end{pmatrix}}_{\mathbf d}.
$$

Primes denote derivatives with respect to $s$; subscripts 0 and $L$ denote the interfaces.

**In the code:** `vectors * f[0]` scales columns of $\mathbf V$, forming $\mathbf V\mathbf F_0$. `bi[:, None] * vectors` scales rows, forming $\mathbf B_i\mathbf V$. `hstack` combines the two weight vectors; `vstack` combines the interfaces.

Finally,

$$
\boxed{\mathbf r=\mathbf V(\mathbf F_0\mathbf a+\mathbf G_0\mathbf b)-\mathbf e_0,\qquad
\mathbf C_{\mathrm{exit}}=\mathbf V(\mathbf F_L\mathbf a+\mathbf G_L\mathbf b).}
$$

The reflected and transmitted coefficients are outputs of this matching problem.


In [ ]:
f_end, g_end, df_end, dg_end = _layer_basis(q, np.array([0., L]), L)
Bi = np.diag(solution.beta_incident / k0)
Bo = np.diag(solution.beta_output / k0)
F0, FL = np.diag(f_end[0]), np.diag(f_end[1])
G0, GL = np.diag(g_end[0]), np.diag(g_end[1])
dF0, dFL = np.diag(df_end[0]), np.diag(df_end[1])
dG0, dGL = np.diag(dg_end[0]), np.diag(dg_end[1])
K = np.block([[V @ dF0 + 1j*Bi @ V @ F0, V @ dG0 + 1j*Bi @ V @ G0],
              [V @ dFL - 1j*Bo @ V @ FL, V @ dGL - 1j*Bo @ V @ GL]])
e0 = np.zeros(N)
e0[config.M] = 1.
rhs = np.r_[2j * Bi @ e0, np.zeros(N)]
weights = np.r_[a, b]

# Inspect solved weights; do not solve a second scattering problem.
assert_allclose(K @ weights, rhs, atol=2e-12)
assert_allclose(V @ (F0 @ a + G0 @ b) - e0, solution.reflection, atol=2e-13)
assert_allclose(V @ (FL @ a + GL @ b), solution.transmission, atol=2e-13)
print(f'Matching matrix: {K.shape}; maximum equation residual: {np.max(abs(K @ weights-rhs)):.3e}')

for row, boundary in enumerate((0., -config.thickness)):
    inside_A = V @ (f_end[row]*a + g_end[row]*b)
    inside_dA = -k0 * V @ (df_end[row]*a + dg_end[row]*b)
    outside_A = solution.harmonics(boundary)[0]
    outside_dA = solution.harmonics(boundary, derivative=1)[0]
    assert_allclose(inside_A, outside_A, atol=2e-12)
    assert_allclose(inside_dA, outside_dA, atol=2e-11)
    print(f'z={boundary:+.2f} um: field jump {np.max(abs(inside_A-outside_A)):.2e}; '
          f'derivative jump {np.max(abs(inside_dA-outside_dA)):.2e} /um')


## 9. What does the numerical boundary solve check?

Entries of $\mathbf K$ can have different scales. `_match_boundaries` divides each row by its largest magnitude, then each column of that row-scaled matrix by its largest magnitude.

With diagonal scale matrices $\mathbf D_r,\mathbf D_c$, it solves

$$
\widetilde{\mathbf K}\mathbf y=\mathbf D_r^{-1}\mathbf d,\qquad
\widetilde{\mathbf K}=\mathbf D_r^{-1}\mathbf K\mathbf D_c^{-1},
\qquad \mathbf c=\mathbf D_c^{-1}\mathbf y.
$$

This **equilibration** changes numerical scales without changing the equations. SciPy solves directly; the code does not explicitly form $\mathbf K^{-1}$.

The reported `condition_number` is $\sigma_{\max}/\sigma_{\min}$ of $\widetilde{\mathbf K}$, obtained by singular-value decomposition. It measures sensitivity of the scaled system, not convergence with harmonic number.

The reported `boundary_residual` is the normwise backward error in the original system,

$$
\rho=\frac{\|\mathbf K\mathbf c-\mathbf d\|_\infty}
{\|\mathbf K\|_\infty\|\mathbf c\|_\infty+\|\mathbf d\|_\infty}.
$$

A small $\rho$ shows that the weights satisfy the finite system accurately. It does **not** establish that finite $M$ resolves the physical grating.

:::{dropdown} Implementation thresholds

A condition number above ${10^{12}}$ triggers a warning. A nonfinite value, or one at least $1/\epsilon_{\mathrm{mach}}$, raises an error. Zero scaling rows/columns are rejected. Nonfinite amplitudes or a backward error above ${10^{-10}}$ raise errors. A singular grazing or trapped-mode problem is not silently perturbed.

:::


In [ ]:
row_scale = np.max(abs(K), axis=1)
row_scaled = K / row_scale[:, None]
column_scale = np.max(abs(row_scaled), axis=0)
K_scaled = row_scaled / column_scale
sigma = linalg.svdvals(K_scaled)
backward_error = linalg.norm(K @ weights-rhs, np.inf) / (
    linalg.norm(K, np.inf)*linalg.norm(weights, np.inf) + linalg.norm(rhs, np.inf))
assert backward_error < 1e-12
assert_allclose(sigma[0]/sigma[-1], solution.condition_number, rtol=1e-10)
print(f'Scaled condition number: {solution.condition_number:.3e}')
print(f'Backward error (explicit matrices): {backward_error:.3e}')
print(f'Backward error (backend):           {solution.boundary_residual:.3e}')


## 10. How are fields, coefficients and power recovered?

`harmonics(z)` evaluates the appropriate exterior expansion or layer basis. Inside the layer, `derivative=1` includes the chain-rule factor $-k_0$, since the basis derivatives were taken with respect to $s$.

`reconstruct(x,z)` evaluates

$$U_M(x,z)=\sum_{m=-M}^{M}F_m(z)e^{ik_{x,m}x}.$$

This is direct matrix multiplication, not FFT propagation. The code's `vectors.T` converts a row of modal weights into a row of harmonic amplitudes. It is a transpose, not a conjugate transpose: the expansion is $\mathbf A=\mathbf V\mathbf w$.

At an output plane $z_{\mathrm{ref}}=-h-\Delta$, with $\Delta\geq0$,

$$\boxed{C_m(z_{\mathrm{ref}})=C_m(z_e) e^{i\beta_{o,m}\Delta}.}$$

Explicitly,

$$C_m(z_{\mathrm{ref}})=F_m(z_{\mathrm{ref}}),\qquad
F_m(z)=C_m(z_{\mathrm{ref}})e^{ik_{z,m}(z-z_{\mathrm{ref}})}$$

in the homogeneous outgoing transmitted region. Thus $F_m(z)$ denotes the function, $C_m(z_{\mathrm{ref}})$ its value at a stated plane, and $C_m(z_e)=C_m(-h)$ the exit-plane value. Inside the layer, use the coupled harmonic solution rather than this single outgoing exponential. A propagating coefficient changes phase; an evanescent coefficient decays.

| Method | Returned shape | Meaning |
| --- | --- | --- |
| `harmonics(z)` | $(n_z,N)$ | Total harmonics, including incidence above the layer |
| `harmonics(z, derivative=1)` | $(n_z,N)$ | Physical derivatives $\partial_z F_m$ |
| `at_reference_plane(z_ref)` | $(N,)$ | One row of harmonic amplitudes |
| `reconstruct(x,z)` | $(n_z,n_x)$ | Complex field samples |

Scalar inputs retain a length-one sample axis, except for `at_reference_plane`. Solution arrays are read-only so sampling and reported coefficients remain consistent.

**Try:** change `distance` below. Compare the spatial field and the propagating/evanescent coefficients. The Fourier check uses the **complex field**, not its magnitude or intensity.


In [ ]:
distance = 0.20  # Micrometres below the exit; try 0.0 and 0.8.
z_ref = -config.thickness - distance
x = np.linspace(-config.period/2, config.period/2, 512, endpoint=False)
C = solution.at_reference_plane(z_ref)
U = solution.reconstruct(x, z_ref)[0]
U_sum = C @ np.exp(1j * solution.kx[:, None] * x)
C_projected = np.mean(U[None, :] * np.exp(-1j * solution.kx[:, None] * x), axis=1)
assert_allclose(C, solution.transmission*np.exp(1j*solution.beta_output*distance), atol=1e-13)
assert_allclose(U, U_sum, atol=1e-13)
assert_allclose(C_projected, C, atol=1e-13)
print(f'Complex-field Fourier projection error: {np.max(abs(C_projected-C)):.3e}')

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5), constrained_layout=True)
axes[0].plot(x, U.real, label='Re U')
axes[0].plot(x, U.imag, label='Im U')
axes[0].set(xlabel=r'$x$ ($\mu$m)', ylabel='Signed field',
            title=fr'Same complex field at $z={z_ref:.2f}$ $\mu$m')
axes[0].legend(fontsize=9)
for kind, color in [('propagating', '#2866a3'), ('evanescent', '#b85c17'), ('grazing', '#79539e')]:
    show = (solution.output_kinds == kind) & (abs(orders) <= 7)
    if np.any(show):
        axes[1].bar(orders[show], abs(C[show]), color=color, label=kind.capitalize())
axes[1].set(xlabel='Order m', ylabel=r'$|C_m|$', title='Field coefficients: displayed orders only',
            xticks=np.arange(-7, 8))
axes[1].legend(fontsize=9)
for ax in axes:
    ax.grid(alpha=.15)
finish_figure(fig)


### Why is efficiency not simply the squared coefficient?

The scalar normal current is $J_z=\operatorname{Im}(U^*\partial_zU)$. A downward propagating component has negative $J_z$, so downward power is proportional to $-J_z$.

Averaging over one period removes cross terms between different orders. Dividing outgoing flux by the incident magnitude $\beta_{i,0}=k_0n_i$ gives

$$
\boxed{\eta_m^{(R)}=\frac{\operatorname{Re}\beta_{i,m}}{\beta_{i,0}}|r_m|^2,\qquad
\eta_m^{(T)}=\frac{\operatorname{Re}\beta_{o,m}}{\beta_{i,0}}|C_m(z_e)|^2.}
$$

The longitudinal factor accounts for power through a plane normal to $z$. A nonzero evanescent amplitude can have zero outgoing normal power. The solver assigns efficiency only to strictly propagating orders.

For this lossless configuration, $R+T=1$, where $R=\sum_m\eta_m^{(R)}$ and $T=\sum_m\eta_m^{(T)}$. This is `solution.power_balance`. Moving the output plane preserves both totals.

**Checkpoint:** which part of a propagating coefficient changes when the plane moves?


In [ ]:
eta_T = solution.beta_output.real / solution.beta_incident[config.M].real * abs(C)**2
assert_allclose(eta_T, solution.transmission_efficiency, atol=1e-13)
R, T = sum(solution.reflection_efficiency), sum(solution.transmission_efficiency)
assert abs(R+T-1) < 1e-10
z_flux = np.array([.1, -.05, -.15, -.25, -.6])
A_flux = solution.harmonics(z_flux)
dA_flux = solution.harmonics(z_flux, derivative=1)
downward_flux = -np.imag(np.sum(A_flux.conj()*dA_flux, axis=1)) / solution.beta_incident[config.M].real
assert_allclose(downward_flux, T, atol=1e-11)
print(f'R={R:.8f}, T={T:.8f}, R+T={R+T:.12f}')
print('Net downward flux above, inside and below the layer:', np.round(downward_flux, 8))


(fmm-validation-checks)=

## 11. Which checks test which claim?

1. **Representation and matching:** the finite solution satisfies its matrix equations and reconstructs its Fourier coefficients. Sections 8--10 check this.
2. **An independent physical limit:** a uniform medium has no scattering and has a known propagation phase.
3. **Truncation convergence:** compare the *same complex orders* at increasing $M$. Energy conservation can be excellent while coefficients are still changing.

For a completely uniform medium of index $n$,

$$r_m=0,\qquad C_m(z_e)=\delta_{m0}e^{ik_0nh},\qquad U(x,z)=e^{-ik_0nz}.$$

The phase of $C_0(z_e)$ reflects the change of reference plane from $0$ to $-h$; it does not imply scattering.

The cell first checks this exact limit, then compares truncations of the patterned example against $M=40$. That finer result is a numerical reference, not an exact solution. Both propagating and evanescent orders are compared.


In [ ]:
uniform_config = replace(config, n_incident=1.2, n_output=1.2, n_ridge=1.2, n_groove=1.2)
uniform = solve_scalar_grating(uniform_config)
expected_t = np.zeros(N, dtype=complex)
expected_t[config.M] = np.exp(1j*k0*1.2*config.thickness)
assert_allclose(uniform.reflection, 0, atol=1e-12)
assert_allclose(uniform.transmission, expected_t, atol=1e-12)
z_test = np.array([.2, -.1, -.6])
assert_allclose(uniform.reconstruct([0.], z_test)[:, 0], np.exp(-1j*k0*1.2*z_test), atol=1e-12)
print('Uniform-medium phase and zero-reflection check: PASS')

compare_orders = np.arange(-3, 4)
fine_M = 40
fine = solve_scalar_grating(replace(config, M=fine_M))
target = np.r_[fine.reflection[compare_orders+fine_M], fine.transmission[compare_orders+fine_M]]
print(' M     max complex-coefficient difference      |R+T-1|')
for truncation in (4, 8, 16, 24):
    trial = solve_scalar_grating(replace(config, M=truncation))
    tested = np.r_[trial.reflection[compare_orders+truncation], trial.transmission[compare_orders+truncation]]
    print(f'{truncation:2d}                 {np.max(abs(tested-target)):.3e}                  '
          f'{abs(trial.power_balance-1):.3e}')


## 12. How does this connect to the absorbing grating in [0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb)--[0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb)?

The public `solve_scalar_grating` accepts a `ScalarGratingConfiguration` with real positive indices. The [absorbing wrapper](wave_optics/absorbing_grating.py) supplies another configuration and calls the **same** `_solve_layer` matching engine.

**The geometry and material coefficients change.** The lossless configuration centres a ridge; the absorbing configuration centres an **air opening**, possibly shifted by $x_c$. With open fraction $D_{\mathrm{open}}$ and absorber permittivity $\varepsilon_a=(n_a+i\kappa)^2$,

$$
\varepsilon_\ell=\varepsilon_a\delta_{\ell0}
+(1-\varepsilon_a)D_{\mathrm{open}}\,\operatorname{sinc}(\ell D_{\mathrm{open}})e^{-i\ell Gx_c}.
$$

**The eigensystem changes.** For a passive absorber, the coupling matrix is generally non-Hermitian. `_solve_layer(..., absorbing=True)` uses `scipy.linalg.eig`, checks the eigenvector condition number and eigensystem residual, and selects roots with nonnegative imaginary parts. Negative eigenvalue imaginary parts within roundoff tolerance are clipped to zero; a materially negative value raises an error. Interface matching and field reconstruction use the same construction.

The wrapper adds an independent volume-loss integral:

$$
\mathcal A=\frac{k_0^2}{\beta_{i,0}d}
\int_{-h}^{0}\int_{-d/2}^{d/2}
\operatorname{Im}\varepsilon(x)\,|U(x,z)|^2\,dx\,dz,
\qquad R+T+\mathcal A=1.
$$

Here $\mathcal A$ is absorbed power, called $A$ in the main notebooks; it is unrelated to the harmonic vector $\mathbf A$.

For a shifted complex material, the loss matrix is $(\mathbf E-\mathbf E^\dagger)/(2i)$, not the elementwise imaginary part of $\mathbf E$. `_volume_absorption` uses it to integrate over $x$ exactly for the retained field, then integrates numerically through the thickness.

`ScalarGratingSolution.power_balance` remains $R+T$ for the absorbing subclass. Its `energy_balance` includes absorption. A scalar loss model still does not supply general vector Maxwell or chrome-on-glass scattering.

:::{dropdown} Absorbing-eigensystem diagnostics

The eigenvector matrix warns at condition number above ${10^{10}}$ and is rejected at $1/\epsilon_{\mathrm{mach}}$ or a nonfinite value. The relative eigen residual is checked against ${10^{-10}}$. The eigenvalue-imaginary-part tolerance is $256\epsilon_{\mathrm{mach}}\max(1,\|\mathbf H\|_2)$. These returned eigensystem diagnostics are computed for the absorbing branch; the lossless branch initializes them to 1 and 0.

:::


In [ ]:
from wave_optics.absorbing_grating import AbsorbingGratingConfiguration, solve_absorbing_grating

absorber_config = AbsorbingGratingConfiguration(M=24)
absorber = solve_absorbing_grating(absorber_config)
H_abs = absorber.coupling_matrix / (2*np.pi/absorber_config.wavelength)**2
assert not np.allclose(H_abs, H_abs.conj().T)
assert abs(absorber.energy_balance-1) < 1e-7
print(f'Absorbing example: R={absorber.reflectance:.6f}, T={absorber.transmittance:.6f}, '
      f'A={absorber.absorption:.6f}')
print(f'R+T+A={absorber.energy_balance:.12f}; eigen residual={absorber.eigen_residual:.2e}')


## 13. A map back to the source

| Code object | Mathematical role | Section |
| --- | --- | --- |
| `_real_scalar`, `_samples` | Validate finite scalars and scalar/1D sample arrays | 1, 10 |
| `ScalarGratingConfiguration` | Specify material, illumination parameters and truncation | 1--2 |
| `permittivity_coefficients` | Exact Fourier coefficients of the material | 2 |
| `refractive_index` | Exact piecewise material; exterior values at horizontal interfaces | 1--2 |
| `_exterior_branch` | Outgoing/decaying roots and grazing classification | 7 |
| `_layer_basis` | Two stable depth solutions and their derivatives | 6 |
| `_match_boundaries` | Equilibrated $2N$-unknown solve and numerical checks | 8--9 |
| `solve_scalar_grating` | Validate the real-index configuration and call the shared engine | 1, 12 |
| `_solve_layer` | Construct $\mathbf H$, find modes, match boundaries, recover $\mathbf r,\mathbf C_{\mathrm{exit}}$ and flux | 3--9 |
| `ScalarGratingSolution` | Consistent read-only arrays, physical $\mathbf Q$ and diagnostics | 10 |
| `harmonics`, `at_reference_plane`, `reconstruct` | Evaluate the existing solution | 10 |

The dense eigensystem and matching solve scale roughly as $O(N^3)$, with $O(N^2)$ matrix storage. Modest truncations are intended. The code rejects nonfinite results and warns when lossless outgoing flux differs from unity by more than $10^{-8}$. These checks do not replace convergence for the quantity you want to predict.

**Checkpoint**

Why do we need two weights for each layer eigenmode? Why does the transmission phase use $z+h$? Why can an order have a nonzero coefficient and zero normal efficiency? Why is a tiny boundary residual insufficient to choose $M$?

:::{dropdown} Check your answers

Each mode obeys a second-order depth equation and must match both interfaces. The factor $z+h$ makes the transmission coefficient reference the exit plane $z=-h$. Evanescent and grazing exterior components have zero real longitudinal propagation factor and zero outgoing normal power. A tiny residual checks the finite solve; increasing $M$ checks how well it resolves the original grating problem. When a propagating output coefficient is moved to another plane, its phase changes while its magnitude stays fixed.

:::

[Foundation index](0_0_Foundations_of_Scalar_Optics.ipynb) · [Physical setup: 0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) · [Fields and orders: 0.6](0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb)




(refs-appendix-1)=

## Further Reading

(ref-appendix-1-scipy-eigh)=

- SciPy documentation, “scipy.linalg.eigh.” [Online]. Available: [source](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eigh.html).

  **Reading notes:** Definitions of the Hermitian and general eigenproblems used in Section 5.

(ref-appendix-1-scipy-eig)=

- SciPy documentation, “scipy.linalg.eig.” [Online]. Available: [source](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eig.html).

(ref-appendix-1-numpy-sinc)=

- NumPy documentation, “numpy.sinc.” [Online]. Available: [source](https://numpy.org/doc/stable/reference/generated/numpy.sinc.html).

  **Reading notes:** The normalized sinc used in the material coefficients and cutoff basis.

(ref-appendix-1-moharam)=

- M. G. Moharam, D. A. Pommet, E. B. Grann, and T. K. Gaylord, “Stable implementation of the rigorous coupled-wave analysis for surface-relief gratings: enhanced transmittance matrix approach,” *JOSA A*, vol. 12, pp. 1077–1086, 1995. [Online]. Available: [DOI](https://doi.org/10.1364/JOSAA.12.001077).

  **Reading notes:** Background on instability from evanescent factors. This implementation uses a single-layer scalar boundary solve, rather than that paper's general electromagnetic algorithm.

(ref-appendix-1-s4)=

- V. Liu and S. Fan, “S4: A free electromagnetic solver for layered periodic structures,” *Computer Physics Communications*, vol. 183, pp. 2233–2244, 2012. [Online]. Available: [PDF](https://web.stanford.edu/group/fan/publication/Liu_ComputerPhysicsCommunications_183_2233_2012.pdf). doi: 10.1016/j.cpc.2012.04.026.

  **Reading notes:** **Access:** Free author-institution PDF; copyright retained. **Read for:** Optional advanced comparison: vector FMM, not our scalar solver.

(ref-appendix-1-maystre)=

- D. Maystre, “Analytic Properties of Diffraction Gratings,” in *Gratings: Theory and Numeric Applications*, 2nd revisited ed., E. Popov, Ed. Institut Fresnel, 2014, ch. 2. [Online]. Available: [PDF](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=33).

  **Reading notes:** **Access:** Free institutional book chapter; not public domain. **Read for:** Read Sections 2.3–2.5: order expansion and energy balance.

## Reference notes

Freely accessible works are not necessarily in the public domain.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.



## Course connections

(course-appendix-1-1)=

[Implementation: `scalar_grating.py`](wave_optics/scalar_grating.py). The source implements the conventions and numerical decisions explained here.

(course-appendix-1-2)=

[Absorbing extension: `absorbing_grating.py`](wave_optics/absorbing_grating.py). Uniform-slab limits, residuals and cutoff convergence test different aspects of the scalar solution.
